# Furniture Dataset Analytics

This notebook provides comprehensive analytics and insights for the furniture recommendation system dataset.

## Overview
- Dataset exploration and cleaning
- Descriptive statistics
- Price analysis and trends
- Category distribution analysis
- Brand and material insights
- Visualizations and dashboards
- Recommendations for business strategy

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Dataset Loading and Initial Exploration

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('../data/sample_furniture_data.csv')
    print(f"Dataset loaded successfully! Shape: {df.shape}")
except FileNotFoundError:
    print("Dataset not found. Creating sample data for analysis...")
    
    # Create sample data
    np.random.seed(42)
    sample_data = {
        'uniq_id': range(1, 101),
        'title': [f'Furniture Item {i}' for i in range(1, 101)],
        'brand': np.random.choice(['IKEA', 'Herman Miller', 'West Elm', 'Pottery Barn', 'CB2', 'Article'], 100),
        'description': [f'High-quality furniture piece {i} with excellent design and functionality' for i in range(1, 101)],
        'price': np.random.uniform(50, 3000, 100).round(2),
        'categories': np.random.choice([
            'Living Room > Sofas', 'Living Room > Coffee Tables', 'Living Room > TV Stands',
            'Bedroom > Beds', 'Bedroom > Nightstands', 'Bedroom > Dressers',
            'Office > Chairs', 'Office > Desks', 'Dining Room > Tables', 'Dining Room > Chairs'
        ], 100),
        'material': np.random.choice(['Wood', 'Metal', 'Fabric', 'Glass', 'Leather', 'Plastic'], 100),
        'color': np.random.choice(['Brown', 'Black', 'White', 'Gray', 'Blue', 'Red', 'Green'], 100),
        'manufacturer': np.random.choice(['IKEA AB', 'Herman Miller Inc.', 'West Elm Corp.'], 100),
        'country_of_origin': np.random.choice(['USA', 'Sweden', 'China', 'Germany', 'Italy'], 100)
    }
    
    df = pd.DataFrame(sample_data)
    print(f"Sample dataset created! Shape: {df.shape}")

# Display basic information
print("\n=== Dataset Info ===")
print(df.info())
print("\n=== First 5 rows ===")
print(df.head())

## 2. Data Cleaning and Preprocessing

In [ ]:
# Check for missing values
print("=== Missing Values ===")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

# Check data types and convert if necessary
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Extract primary and secondary categories
df['primary_category'] = df['categories'].str.split(' > ').str[0]
df['secondary_category'] = df['categories'].str.split(' > ').str[1]

# Handle missing values
df = df.fillna('')

print("\n=== Data after preprocessing ===")
print(f"Shape: {df.shape}")
print(f"Price column stats:")
print(df['price'].describe())

## 3. Descriptive Statistics

In [ ]:
# Overall dataset statistics
print("=== DATASET OVERVIEW ===")
print(f"Total Products: {len(df)}")
print(f"Unique Brands: {df['brand'].nunique()}")
print(f"Unique Categories: {df['primary_category'].nunique()}")
print(f"Unique Materials: {df['material'].nunique()}")
print(f"Price Range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
print(f"Average Price: ${df['price'].mean():.2f}")
print(f"Median Price: ${df['price'].median():.2f}")

# Category distribution
print("\n=== CATEGORY DISTRIBUTION ===")
category_dist = df['primary_category'].value_counts()
print(category_dist)

# Brand distribution
print("\n=== BRAND DISTRIBUTION ===")
brand_dist = df['brand'].value_counts()
print(brand_dist)

# Material distribution
print("\n=== MATERIAL DISTRIBUTION ===")
material_dist = df['material'].value_counts()
print(material_dist)

## 4. Price Analysis

In [ ]:
# Price distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Price histogram
axes[0, 0].hist(df['price'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Price Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Price by category boxplot
df.boxplot(column='price', by='primary_category', ax=axes[0, 1])
axes[0, 1].set_title('Price Distribution by Category', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Category')
axes[0, 1].set_ylabel('Price ($)')
plt.setp(axes[0, 1].xaxis.get_majorticklabels(), rotation=45)

# Price by brand
brand_prices = df.groupby('brand')['price'].mean().sort_values(ascending=False)
axes[1, 0].bar(brand_prices.index, brand_prices.values, color='lightgreen')
axes[1, 0].set_title('Average Price by Brand', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Brand')
axes[1, 0].set_ylabel('Average Price ($)')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45)

# Price by material
material_prices = df.groupby('material')['price'].mean().sort_values(ascending=False)
axes[1, 1].bar(material_prices.index, material_prices.values, color='salmon')
axes[1, 1].set_title('Average Price by Material', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Material')
axes[1, 1].set_ylabel('Average Price ($)')
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

# Price statistics by category
print("\n=== PRICE STATISTICS BY CATEGORY ===")
price_stats = df.groupby('primary_category')['price'].agg(['mean', 'median', 'min', 'max', 'std', 'count'])
price_stats = price_stats.round(2)
print(price_stats)

## 5. Interactive Visualizations with Plotly

In [ ]:
# Interactive price distribution
fig_price_hist = px.histogram(df, x='price', nbins=20, 
                             title='Interactive Price Distribution',
                             labels={'price': 'Price ($)', 'count': 'Number of Products'})
fig_price_hist.update_layout(showlegend=False)
fig_price_hist.show()

# Category distribution pie chart
category_counts = df['primary_category'].value_counts()
fig_pie = px.pie(values=category_counts.values, names=category_counts.index,
                 title='Product Distribution by Category')
fig_pie.show()

# Price vs Category scatter plot with brand information
fig_scatter = px.scatter(df, x='primary_category', y='price', color='brand',
                        title='Price Distribution by Category and Brand',
                        labels={'price': 'Price ($)', 'primary_category': 'Category'})
fig_scatter.update_layout(xaxis_tickangle=-45)
fig_scatter.show()

## 6. Advanced Analytics

In [ ]:
# Price segments analysis
def categorize_price(price):
    if price < 200:
        return 'Budget (< $200)'
    elif price < 500:
        return 'Mid-range ($200-500)'
    elif price < 1000:
        return 'Premium ($500-1000)'
    else:
        return 'Luxury ($1000+)'

df['price_segment'] = df['price'].apply(categorize_price)

# Price segment distribution
print("=== PRICE SEGMENT ANALYSIS ===")
segment_dist = df['price_segment'].value_counts()
print(segment_dist)
print(f"\nPercentage distribution:")
print((segment_dist / len(df) * 100).round(2))

# Cross-tabulation analysis
print("\n=== CATEGORY vs PRICE SEGMENT ===")
crosstab = pd.crosstab(df['primary_category'], df['price_segment'])
print(crosstab)

# Brand vs Category analysis
print("\n=== BRAND vs CATEGORY ===")
brand_category = pd.crosstab(df['brand'], df['primary_category'])
print(brand_category)

## 7. Correlation Analysis

In [ ]:
# Create dummy variables for categorical data
df_numeric = pd.get_dummies(df[['price', 'primary_category', 'brand', 'material']], prefix=['cat', 'brand', 'mat'])

# Correlation matrix for price with other features
price_correlations = df_numeric.corrwith(df_numeric['price']).abs().sort_values(ascending=False)

print("=== PRICE CORRELATIONS ===")
print("Top correlations with price:")
print(price_correlations.head(10))

# Visualize correlations
plt.figure(figsize=(12, 8))
top_correlations = price_correlations.head(15)
plt.barh(range(len(top_correlations)), top_correlations.values)
plt.yticks(range(len(top_correlations)), top_correlations.index)
plt.xlabel('Correlation with Price')
plt.title('Feature Correlations with Price')
plt.tight_layout()
plt.show()

## 8. Market Insights and Recommendations

In [ ]:
# Key insights from the data
print("=== KEY MARKET INSIGHTS ===")

# 1. Most profitable categories
category_revenue = df.groupby('primary_category').agg({
    'price': ['mean', 'count', 'sum']
}).round(2)
category_revenue.columns = ['avg_price', 'product_count', 'total_value']
category_revenue = category_revenue.sort_values('total_value', ascending=False)

print("\n1. Category Performance:")
print(category_revenue)

# 2. Brand positioning
brand_positioning = df.groupby('brand').agg({
    'price': ['mean', 'min', 'max', 'count']
}).round(2)
brand_positioning.columns = ['avg_price', 'min_price', 'max_price', 'product_count']
brand_positioning = brand_positioning.sort_values('avg_price', ascending=False)

print("\n2. Brand Positioning:")
print(brand_positioning)

# 3. Material premium analysis
material_premium = df.groupby('material')['price'].mean().sort_values(ascending=False)
print("\n3. Material Premium Analysis:")
print(material_premium)

# 4. Market gaps analysis
print("\n4. Market Gaps and Opportunities:")

# Find underrepresented categories
low_count_categories = category_revenue[category_revenue['product_count'] < 5]
if len(low_count_categories) > 0:
    print("Underrepresented categories with growth potential:")
    print(low_count_categories.index.tolist())

# Find price gaps
price_ranges = df.groupby('primary_category')['price'].apply(lambda x: x.max() - x.min())
large_price_gaps = price_ranges[price_ranges > price_ranges.mean()]
print(f"\nCategories with large price ranges (potential for product diversification):")
print(large_price_gaps.sort_values(ascending=False))

## 9. Recommendations for Business Strategy

In [ ]:
print("=== BUSINESS RECOMMENDATIONS ===")

# Generate actionable insights
recommendations = []

# 1. Top performing categories
top_category = category_revenue.index[0]
recommendations.append(f"Focus marketing efforts on {top_category} - highest revenue category")

# 2. Price optimization
avg_market_price = df['price'].mean()
underpriced_categories = category_revenue[category_revenue['avg_price'] < avg_market_price * 0.8]
if len(underpriced_categories) > 0:
    recommendations.append(f"Consider price increases for: {', '.join(underpriced_categories.index)}")

# 3. Brand strategy
premium_brands = brand_positioning[brand_positioning['avg_price'] > avg_market_price]
budget_brands = brand_positioning[brand_positioning['avg_price'] < avg_market_price * 0.7]

if len(premium_brands) > 0:
    recommendations.append(f"Premium positioning for: {', '.join(premium_brands.index)}")
if len(budget_brands) > 0:
    recommendations.append(f"Budget-friendly positioning for: {', '.join(budget_brands.index)}")

# 4. Product gaps
all_categories = df['primary_category'].unique()
all_materials = df['material'].unique()
category_material_combinations = df.groupby(['primary_category', 'material']).size()

# Print recommendations
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

print("\n=== DATA QUALITY INSIGHTS ===")
print(f"Dataset completeness: {(1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100:.1f}%")
print(f"Price data quality: {df['price'].notna().sum() / len(df) * 100:.1f}% complete")
print(f"Category coverage: {df['primary_category'].nunique()} unique categories")
print(f"Brand diversity: {df['brand'].nunique()} unique brands")

## 10. Export Results

In [ ]:
# Create summary report
summary_report = {
    'dataset_overview': {
        'total_products': len(df),
        'unique_brands': df['brand'].nunique(),
        'unique_categories': df['primary_category'].nunique(),
        'price_range': [float(df['price'].min()), float(df['price'].max())],
        'average_price': float(df['price'].mean())
    },
    'category_performance': category_revenue.to_dict(),
    'brand_positioning': brand_positioning.to_dict(),
    'recommendations': recommendations
}

# Save processed data
df.to_csv('../data/processed_furniture_data.csv', index=False)
print("Processed data saved to '../data/processed_furniture_data.csv'")

# Save analytics summary
import json
with open('../data/analytics_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2, default=str)
print("Analytics summary saved to '../data/analytics_summary.json'")

print("\n=== ANALYSIS COMPLETE ===")
print("This comprehensive analysis provides insights for:")
print("- Product recommendation algorithms")
print("- Pricing strategies")
print("- Inventory management")
print("- Marketing focus areas")
print("- Business growth opportunities")